In [1]:
import pandas as pd
import numpy as np

In [2]:
final = pd.read_csv("../data/processed/cic_final_merged.csv")

final['Label'].value_counts()

C:\Users\Systems\AppData\Local\Temp\ipykernel_23952\3679454291.py:1: DtypeWarning: Columns (1,2,4,7,85) have mixed types. Specify dtype option on import or set low_memory=False.
  final = pd.read_csv("../data/processed/cic_final_merged.csv")


Label
DrDoS_UDP    470202
BENIGN       341896
Syn          237337
UDP-lag       54981
WebDDoS          62
Name: count, dtype: int64

In [3]:
final.shape

(1104478, 88)

In [4]:
final.columns = final.columns.str.strip()
final.columns = final.columns.str.replace(" ", "_")
final.columns = final.columns.str.replace("/", "_")
final.columns = final.columns.str.replace(".", "_")

In [5]:
final.dtypes.value_counts()

float64    49
int64      33
object      6
Name: count, dtype: int64

In [6]:
for col in final.columns:
    if col != "Label" and final[col].dtype == "object":
        final[col] = pd.to_numeric(final[col], errors="coerce")

In [7]:
final.replace([np.inf, -np.inf], np.nan, inplace=True)
final.fillna(0, inplace=True)

In [8]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
final["Label_encoded"] = le.fit_transform(final["Label"])

le.classes_

array(['BENIGN', 'DrDoS_UDP', 'Syn', 'UDP-lag', 'WebDDoS'], dtype=object)

In [9]:
final.to_csv("../data/processed/cic_final_clean.csv", index=False)

In [10]:
final.dtypes.value_counts()

float64    54
int64      34
object      1
Name: count, dtype: int64

In [11]:
final.head()

,Unnamed:_0,Flow_ID,Source_IP,Source_Port,Destination_IP,Destination_Port,Protocol,Timestamp,Flow_Duration,Total_Fwd_Packets,...,Active_Max,Active_Min,Idle_Mean,Idle_Std,Idle_Max,Idle_Min,SimillarHTTP,Inbound,Label,Label_encoded
0,69186.0,0.0,0.0,23568.0,0.0,23568,6.0,0.0,105601044,16,...,92.0,1.0,1.508584e+07,4.970854e+06,23224044.0,7835990.0,0.0,1.0,Syn,2
1,175890.0,0.0,0.0,5620.0,0.0,9038,6.0,0.0,48,2,...,0.0,0.0,0.000000e+00,0.000000e+00,0.0,0.0,0.0,1.0,Syn,2
2,382355.0,0.0,0.0,60611.0,0.0,14717,6.0,0.0,51,2,...,0.0,0.0,0.000000e+00,0.000000e+00,0.0,0.0,0.0,1.0,Syn,2
3,84988.0,0.0,0.0,7372.0,0.0,24770,6.0,0.0,102,2,...,0.0,0.0,0.000000e+00,0.000000e+00,0.0,0.0,0.0,1.0,Syn,2
4,686595.0,0.0,0.0,14377.0,0.0,14377,6.0,0.0,83234013,8,...,1.0,1.0,2.774467e+07,1.704875e+07,46683960.0,13623423.0,0.0,1.0,Syn,2


In [25]:
cols_to_drop = [
    "Unnamed:_0", 
    "Flow_ID", 
    "Source_IP", 
    "Destination_IP",
    "Timestamp",
    "SimillarHTTP"
]

final = final.drop(columns=[c for c in cols_to_drop if c in final.columns])
final.shape

(1104478, 83)

In [13]:
final = final[final["Label"] != "WebDDoS"]
final["Label"].value_counts()

Label
DrDoS_UDP    470202
BENIGN       341896
Syn          237337
UDP-lag       54981
Name: count, dtype: int64

In [28]:
X = final.drop(["Label", "Label_encoded"], axis=1)
y = final["Label_encoded"]

In [29]:
X.shape, y.shape

((1104478, 81), (1104478,))

In [30]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [31]:
import os, joblib
os.makedirs("../models", exist_ok=True)
joblib.dump(scaler, "../models/standard_scaler.pkl")

['../models/standard_scaler.pkl']

In [32]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

In [33]:
y_train.shape, y_test.shape

((883582,), (220896,))

In [34]:
X_train.shape, X_test.shape

((883582, 81), (220896, 81))

In [35]:
X_train.shape

(883582, 81)

In [22]:
final = pd.read_csv("../data/processed/cic_final_clean.csv")

In [23]:
final["packet_rate"] = (
    final[" Total Fwd Packets"] + final[" Total Backward Packets"]
) / (final["Flow Duration"] + 1e-6)

KeyError: ' Total Fwd Packets'

In [24]:
list(final.columns)

['Unnamed:_0',
 'Flow_ID',
 'Source_IP',
 'Source_Port',
 'Destination_IP',
 'Destination_Port',
 'Protocol',
 'Timestamp',
 'Flow_Duration',
 'Total_Fwd_Packets',
 'Total_Backward_Packets',
 'Total_Length_of_Fwd_Packets',
 'Total_Length_of_Bwd_Packets',
 'Fwd_Packet_Length_Max',
 'Fwd_Packet_Length_Min',
 'Fwd_Packet_Length_Mean',
 'Fwd_Packet_Length_Std',
 'Bwd_Packet_Length_Max',
 'Bwd_Packet_Length_Min',
 'Bwd_Packet_Length_Mean',
 'Bwd_Packet_Length_Std',
 'Flow_Bytes_s',
 'Flow_Packets_s',
 'Flow_IAT_Mean',
 'Flow_IAT_Std',
 'Flow_IAT_Max',
 'Flow_IAT_Min',
 'Fwd_IAT_Total',
 'Fwd_IAT_Mean',
 'Fwd_IAT_Std',
 'Fwd_IAT_Max',
 'Fwd_IAT_Min',
 'Bwd_IAT_Total',
 'Bwd_IAT_Mean',
 'Bwd_IAT_Std',
 'Bwd_IAT_Max',
 'Bwd_IAT_Min',
 'Fwd_PSH_Flags',
 'Bwd_PSH_Flags',
 'Fwd_URG_Flags',
 'Bwd_URG_Flags',
 'Fwd_Header_Length',
 'Bwd_Header_Length',
 'Fwd_Packets_s',
 'Bwd_Packets_s',
 'Min_Packet_Length',
 'Max_Packet_Length',
 'Packet_Length_Mean',
 'Packet_Length_Std',
 'Packet_Length_Varianc

In [26]:
final.head()

,Source_Port,Destination_Port,Protocol,Flow_Duration,Total_Fwd_Packets,Total_Backward_Packets,Total_Length_of_Fwd_Packets,Total_Length_of_Bwd_Packets,Fwd_Packet_Length_Max,Fwd_Packet_Length_Min,...,Active_Std,Active_Max,Active_Min,Idle_Mean,Idle_Std,Idle_Max,Idle_Min,Inbound,Label,Label_encoded
0,23568.0,23568,6.0,105601044,16,2,0.0,0.0,0.0,0.0,...,40.990243,92.0,1.0,1.508584e+07,4.970854e+06,23224044.0,7835990.0,1.0,Syn,2
1,5620.0,9038,6.0,48,2,0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.000000e+00,0.000000e+00,0.0,0.0,1.0,Syn,2
2,60611.0,14717,6.0,51,2,0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.000000e+00,0.000000e+00,0.0,0.0,1.0,Syn,2
3,7372.0,24770,6.0,102,2,2,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.000000e+00,0.000000e+00,0.0,0.0,1.0,Syn,2
4,14377.0,14377,6.0,83234013,8,0,0.0,0.0,0.0,0.0,...,0.000000,1.0,1.0,2.774467e+07,1.704875e+07,46683960.0,13623423.0,1.0,Syn,2


In [27]:
list(final.columns)

['Source_Port',
 'Destination_Port',
 'Protocol',
 'Flow_Duration',
 'Total_Fwd_Packets',
 'Total_Backward_Packets',
 'Total_Length_of_Fwd_Packets',
 'Total_Length_of_Bwd_Packets',
 'Fwd_Packet_Length_Max',
 'Fwd_Packet_Length_Min',
 'Fwd_Packet_Length_Mean',
 'Fwd_Packet_Length_Std',
 'Bwd_Packet_Length_Max',
 'Bwd_Packet_Length_Min',
 'Bwd_Packet_Length_Mean',
 'Bwd_Packet_Length_Std',
 'Flow_Bytes_s',
 'Flow_Packets_s',
 'Flow_IAT_Mean',
 'Flow_IAT_Std',
 'Flow_IAT_Max',
 'Flow_IAT_Min',
 'Fwd_IAT_Total',
 'Fwd_IAT_Mean',
 'Fwd_IAT_Std',
 'Fwd_IAT_Max',
 'Fwd_IAT_Min',
 'Bwd_IAT_Total',
 'Bwd_IAT_Mean',
 'Bwd_IAT_Std',
 'Bwd_IAT_Max',
 'Bwd_IAT_Min',
 'Fwd_PSH_Flags',
 'Bwd_PSH_Flags',
 'Fwd_URG_Flags',
 'Bwd_URG_Flags',
 'Fwd_Header_Length',
 'Bwd_Header_Length',
 'Fwd_Packets_s',
 'Bwd_Packets_s',
 'Min_Packet_Length',
 'Max_Packet_Length',
 'Packet_Length_Mean',
 'Packet_Length_Std',
 'Packet_Length_Variance',
 'FIN_Flag_Count',
 'SYN_Flag_Count',
 'RST_Flag_Count',
 'PSH_Flag_Co

In [36]:
list(final.columns)

['Source_Port',
 'Destination_Port',
 'Protocol',
 'Flow_Duration',
 'Total_Fwd_Packets',
 'Total_Backward_Packets',
 'Total_Length_of_Fwd_Packets',
 'Total_Length_of_Bwd_Packets',
 'Fwd_Packet_Length_Max',
 'Fwd_Packet_Length_Min',
 'Fwd_Packet_Length_Mean',
 'Fwd_Packet_Length_Std',
 'Bwd_Packet_Length_Max',
 'Bwd_Packet_Length_Min',
 'Bwd_Packet_Length_Mean',
 'Bwd_Packet_Length_Std',
 'Flow_Bytes_s',
 'Flow_Packets_s',
 'Flow_IAT_Mean',
 'Flow_IAT_Std',
 'Flow_IAT_Max',
 'Flow_IAT_Min',
 'Fwd_IAT_Total',
 'Fwd_IAT_Mean',
 'Fwd_IAT_Std',
 'Fwd_IAT_Max',
 'Fwd_IAT_Min',
 'Bwd_IAT_Total',
 'Bwd_IAT_Mean',
 'Bwd_IAT_Std',
 'Bwd_IAT_Max',
 'Bwd_IAT_Min',
 'Fwd_PSH_Flags',
 'Bwd_PSH_Flags',
 'Fwd_URG_Flags',
 'Bwd_URG_Flags',
 'Fwd_Header_Length',
 'Bwd_Header_Length',
 'Fwd_Packets_s',
 'Bwd_Packets_s',
 'Min_Packet_Length',
 'Max_Packet_Length',
 'Packet_Length_Mean',
 'Packet_Length_Std',
 'Packet_Length_Variance',
 'FIN_Flag_Count',
 'SYN_Flag_Count',
 'RST_Flag_Count',
 'PSH_Flag_Co

In [37]:
final["packet_rate"] = (
    final[" Total Fwd Packets"] + final[" Total Backward Packets"]
) / (final["Flow Duration"] + 1e-6)

KeyError: ' Total Fwd Packets'

In [38]:
final["packet_rate"] = (
    final["Total Fwd Packets"] + final["Total Backward Packets"]
) / (final["Flow Duration"] + 1e-6)

KeyError: 'Total Fwd Packets'

In [39]:
cols_to_drop = [
    "Unnamed:_0", 
    "Flow_ID", 
    "Source_IP", 
    "Destination_IP",
    "Timestamp",
    "SimillarHTTP"
]

final = final.drop(columns=[c for c in cols_to_drop if c in final.columns])

In [40]:
import numpy as np

# 1. Packet Rate
final["packet_rate"] = (
    final["Total_Fwd_Packets"] + final["Total_Backward_Packets"]
) / (final["Flow_Duration"] + 1e-6)

# 2. Byte Rate
final["byte_rate"] = (
    final["Total_Length_of_Fwd_Packets"] + final["Total_Length_of_Bwd_Packets"]
) / (final["Flow_Duration"] + 1e-6)

# 3. Burstiness
final["burstiness"] = final["Fwd_IAT_Std"] / (final["Fwd_IAT_Mean"] + 1e-6)

# 4. Flow Size
final["flow_size"] = (
    final["Total_Length_of_Fwd_Packets"] + final["Total_Length_of_Bwd_Packets"]
)

# 5. Average Payload Size
final["avg_payload_size"] = (
    final["Total_Length_of_Fwd_Packets"] + final["Total_Length_of_Bwd_Packets"]
) / (
    final["Total_Fwd_Packets"] + final["Total_Backward_Packets"] + 1e-6
)

# 6. Packet Size Variance
final["packet_size_var"] = final["Packet_Length_Variance"]

# 7. Fwd/Bwd Packet Ratio
final["fwd_bwd_ratio"] = (
    final["Total_Fwd_Packets"] / (final["Total_Backward_Packets"] + 1e-6)
)

# 8. Combined IAT Mean
final["iat_mean_combined"] = (
    final["Flow_IAT_Mean"] + final["Fwd_IAT_Mean"] + final["Bwd_IAT_Mean"]
) / 3

# 9. Total Flags
final["total_flags"] = (
    final["SYN_Flag_Count"] +
    final["ACK_Flag_Count"] +
    final["PSH_Flag_Count"] +
    final["URG_Flag_Count"] +
    final["RST_Flag_Count"]
)

# 10. Flag Intensity
final["flag_intensity"] = final["total_flags"] / (final["Flow_Duration"] + 1e-6)

# 11. SYN Dominance
final["syn_dominance"] = final["SYN_Flag_Count"] / (final["total_flags"] + 1e-6)

# 12. Packet Energy
final["packet_energy"] = final["Packet_Length_Mean"] * final["burstiness"]

In [41]:
final.to_csv("../data/processed/cic_final_behavior.csv", index=False)